In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.chdir('./test_time_gd/')
from kv_dataset_utils import generate_sequence, get_extra_chars, BASE_KV_ALPHABET

In [ ]:
num_kv_pairs = 256
k_length = 2
v_length = 2
n_segments = 1
min_segment_len = 0
max_segment_len = 0

kv_vocab_size = 62

kv_alphabet = BASE_KV_ALPHABET + get_extra_chars(kv_vocab_size)

sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len, kv_alphabet)

sample


In [ ]:
print('KV pairs length:', (k_length + v_length + 3)*num_kv_pairs)
print('~Total length:', n_segments * (min_segment_len + max_segment_len)/2)

In [ ]:
sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                           min_segment_len, max_segment_len, kv_alphabet)
sample

In [ ]:
from datasets import Dataset, DatasetDict
from tqdm import tqdm

num_samples = 1_000_000

data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                               min_segment_len, max_segment_len, kv_alphabet)
    data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]
data = Dataset.from_list(data)

num_samples = 5_000

valid_data = []
for _ in tqdm(range(num_samples), total=num_samples):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                               min_segment_len, max_segment_len, kv_alphabet)
    valid_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]

valid_data = Dataset.from_list(valid_data)
test_data = []
for _ in tqdm(range(num_samples * 2), total=num_samples * 2):
    sample = generate_sequence(num_kv_pairs, k_length, v_length, n_segments,
                               min_segment_len, max_segment_len, kv_alphabet)
    test_data += [{
        'context': sample['context'],
        'query': sample['query'],
        'target': sample['target'],
    }]

test_data = Dataset.from_list(test_data)

dataset = DatasetDict({'train': data, 'valid': valid_data, 'test': test_data})

In [ ]:
dataset.push_to_hub('irodkin/kv_retrieval', config_name=f"N{num_kv_pairs}-K{k_length}V{v_length}-V{kv_vocab_size}")